# D.R.O.N.A. — Notebook 07: ACT Gesture Policy (LeRobot)

Trains an **Action Chunking Transformer (ACT)** imitation policy on the gesture demonstrations — Research Contribution **C3**. Needs a **GPU** (Colab T4 / Kaggle T4). ~15–30 min. Output: `data/checkpoints/act/` — drop it into the repo and the `PolicyRouter` / web-twin / ROS2 `policy_node` load it automatically.

**Order:** run the cells top to bottom. Edit `REPO_URL` in cell 1 first.

## 1. Setup (edit `REPO_URL`)

In [ ]:
# ===========================================================================
#  D.R.O.N.A. — Colab / Kaggle setup.  *** RUN THIS CELL FIRST ***
#  Works on Google Colab and Kaggle. See docs/COLAB_TRAINING_GUIDE.md.
# ===========================================================================
import os, sys, subprocess, pathlib

# 1) GPU check ---------------------------------------------------------------
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "!!! NO GPU SELECTED.\n"
             "    Colab : Runtime > Change runtime type > T4 GPU\n"
             "    Kaggle: right panel > Settings > Accelerator > GPU T4 x2")

# 2) Get the D.R.O.N.A. repo -------------------------------------------------
#    EDIT REPO_URL to your GitHub repo. Private repo? use a read token:
#      https://<TOKEN>@github.com/<user>/D.R.O.N.A.git
#    OR leave it as-is and instead UPLOAD a zip of the project (Colab: Files panel)
#    / attach it as a Kaggle dataset named 'drona' — the search loop finds it.
REPO_URL = "https://github.com/<your-username>/D.R.O.N.A.git"   # <-- EDIT ME

def _is_repo(p):
    return pathlib.Path(p, "drona", "__init__.py").is_file()

search = ["D.R.O.N.A", ".", "/content/D.R.O.N.A",
          "/kaggle/working/D.R.O.N.A", "/kaggle/input/drona/D.R.O.N.A", "/kaggle/input/drona"]
repo = next((p for p in search if _is_repo(p)), None)
if repo is None and "<your-username>" not in REPO_URL:
    dest = "/content/D.R.O.N.A" if pathlib.Path("/content").is_dir() else "D.R.O.N.A"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
    repo = dest
assert repo and _is_repo(repo), (
    "Repo not found. Set REPO_URL to your GitHub URL, OR upload/attach the project, "
    "then re-run. See docs/COLAB_TRAINING_GUIDE.md.")
os.chdir(repo)
print("repo:", os.getcwd())

# 3) Install the drona package (light deps; training deps are installed below)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=False)
print("setup complete -- continue to the next cell")

## 2. Install LeRobot

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/lerobot.git"], check=True)
print("lerobot installed")

## 3. Generate demonstrations + build the LeRobotDataset
Self-contained: regenerates the keyframe demonstrations (no external data needed).

In [ ]:
subprocess.run([sys.executable, "scripts/collect_demonstrations.py",
                "--episodes", "25", "--out-dir", "data/demonstrations"], check=True)

import pathlib
from drona.interaction.demonstration import DemonstrationDataset
from drona.interaction.lerobot_dataset import build_lerobot_dataset, LEROBOT_FEATURES
demo = DemonstrationDataset.load_jsonl(pathlib.Path("data/demonstrations/demonstrations.jsonl"))
print("frames:", demo.total_frames, "| episodes:", len(demo.episodes))
lerobot_ds = build_lerobot_dataset(demo, repo_id="drona/gestures", fps=20,
                                   root="data/lerobot/drona_gestures")
print("LeRobotDataset ready at data/lerobot/drona_gestures")

## 4. Train ACT
Uses the LeRobot training CLI (its Python API names drift between versions). Lower `--steps` for a quick smoke run; raise it for quality.

In [ ]:
!python -m lerobot.scripts.train \
    --dataset.repo_id=drona/gestures \
    --dataset.root=data/lerobot/drona_gestures \
    --policy.type=act \
    --policy.chunk_size=10 \
    --policy.n_action_steps=10 \
    --policy.dim_model=256 \
    --batch_size=32 \
    --steps=20000 \
    --output_dir=data/checkpoints/act \
    --job_name=drona_act \
    --device=cuda \
    --wandb.enable=false

## 5. Evaluate: ACT vs keyframe baseline (success / jerk)

In [ ]:
import json
from drona.interaction.act_policy import KeyframePolicy, LeRobotACTPolicy
from drona.interaction.sim_eval import compare_policies

CKPT = "data/checkpoints/act"
def act_factory(g):
    try:
        return LeRobotACTPolicy(f"{CKPT}/{g}", device="cuda")
    except Exception:
        return LeRobotACTPolicy(CKPT, device="cuda")
res = compare_policies(base_factory=lambda g: KeyframePolicy(g), other_factory=act_factory)
print(json.dumps(res["delta"], indent=2))
print("ACT smoother than keyframe:", res["smoother"])

## 6. Download the trained ACT checkpoint

In [ ]:
# === Download your trained model back to your computer ===
# Edit SRC if you trained into a different directory.
import shutil, pathlib
SRC = "data/checkpoints/act"
base = "/content" if pathlib.Path("/content").is_dir() else "/kaggle/working"
zip_path = shutil.make_archive(f"{base}/drona_act_checkpoints", "zip", SRC)
print("zipped:", zip_path, "->", round(pathlib.Path(zip_path).stat().st_size/1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zip_path)                      # Colab: browser download starts
except Exception:
    print("Kaggle: open the right-hand 'Output' tab, find the .zip, and click Download.")
print("\nNext (on your PC): unzip into the repo so the paths match:\n  unzip drona_act_checkpoints.zip -d data/checkpoints/act")